# 적용 아이디어 추가 분석

* 목적: README의 성장·퍼널·상품 제안을 뒷받침하는 집계
* 기본 기간: 2023-01-01 이상, 2026-01-01 미만(UTC)
* 관측 종료: 2026-02-28 00:00 UTC
* 구성: 계산 코드 뒤 집계표 표시
* 성장: 취소 제외 주문 기준
* 상품: Complete 품목 기준
* 퍼널: 구매 이벤트 세션과 세션 내 ID 상태 확인
* 실험·로그 보강: 제안 단계
* 아래 집계 결과의 실행 효과 해석 불가

In [1]:
from pathlib import Path
import sys

import pandas as pd
import pyarrow.parquet as pq

code_dir = next(
    (candidate for parent in (Path.cwd(), *Path.cwd().parents)
     for candidate in (parent, parent / "code")
     if (candidate / "ideas" / "analyze_ideas.py").is_file()
     and (candidate / "app" / "data_mart").is_dir()),
    None,
)
if code_dir is None:
    raise FileNotFoundError("프로젝트 루트 또는 code/ideas, code/notebooks 폴더에서 실행해 주세요.")
sys.path.insert(0, str(code_dir / "app"))
from data_mart.analysis_contract import (
    GROWTH_STATUSES, OBSERVATION_END, PURCHASE_STATUS, default_period,
)

parquet_dir = code_dir / "app" / "data" / "parquet"
required = ("fact_order", "mart_funnel_session", "df_e", "mart_product_mix_base")
missing = [name + ".parquet" for name in required
           if not (parquet_dir / (name + ".parquet")).is_file()]
if missing:
    raise FileNotFoundError(
        "분석에 필요한 Parquet가 없습니다: " + ", ".join(missing)
        + "\nREADME의 BigQuery 준비를 마친 뒤 code/pipeline 폴더에서 "
        + "다음 두 명령을 순서대로 실행해 주세요:\n"
        + "python run_sql.py\npython export_parquet.py"
    )
start_year, end_year = default_period()
start = pd.Timestamp(f"{start_year}-01-01", tz="UTC")
end = pd.Timestamp(f"{end_year + 1}-01-01", tz="UTC")

## 성장: 첫 주문과 반복 주문

* 취소 제외 주문 전체에서 고객별 첫 주문 판정
* 첫 주문 판정 후 분석 기간의 주문 건수·금액 비교

In [2]:
def growth_order_type(parquet_dir, start, end):
    f = pd.read_parquet(parquet_dir / "fact_order.parquet", columns=["o_id", "u_id", "order_date", "order_status", "order_revenue"])
    f["order_date"] = pd.to_datetime(f["order_date"], utc=True)
    f = f[f["order_status"].isin(GROWTH_STATUSES) & f["order_date"].le(OBSERVATION_END)].copy()

    # 대시보드 성장 마트(02_growth.sql)와 같은 기준:
    # Cancelled를 뺀 주문 안에서 고객별 순번을 다시 매겨 1번만 첫 주문으로 본다.
    f = f.sort_values(["u_id", "order_date", "o_id"])
    f["order_type"] = (f.groupby("u_id").cumcount() == 0).map({True: "first", False: "repeat"})

    d = f[f["order_date"].ge(start) & f["order_date"].lt(end)].copy()
    d["year"] = d["order_date"].dt.year
    t = d.pivot_table(index="year", columns="order_type", values="o_id", aggfunc="count").reset_index()
    amt = d.pivot_table(index="year", columns="order_type", values="order_revenue", aggfunc="sum").reset_index(drop=True)
    t["orders"] = t["first"] + t["repeat"]
    t["repeat_share_pct"] = t["repeat"] / t["orders"] * 100
    # 금액 기준 비중(대시보드 New vs Repeat Order Amount)과 건수 기준 비중 비교
    t["repeat_amount_share_pct"] = amt["repeat"] / (amt["first"] + amt["repeat"]) * 100
    t["first_amount_per_order"] = amt["first"] / t["first"]
    t["repeat_amount_per_order"] = amt["repeat"] / t["repeat"]
    return t[["year", "orders", "first", "repeat", "repeat_share_pct",
              "repeat_amount_share_pct", "first_amount_per_order", "repeat_amount_per_order"]]

In [3]:
growth_table = growth_order_type(parquet_dir, start, end)
growth_table

order_type,year,orders,first,repeat,repeat_share_pct,repeat_amount_share_pct,first_amount_per_order,repeat_amount_per_order
0,2023,14403,10560,3843,26.681941,26.455089,86.880654,85.876279
1,2024,21589,14701,6888,31.905137,32.080363,86.38433,87.08285
2,2025,36250,21886,14364,39.624828,39.671463,86.141224,86.309275


## 퍼널: 구매 경로와 ID 연결

* 구매 이벤트 세션의 경로 유형 확인
* 세션 시작일 기준 ID 유지 상태 확인
* 이벤트 원본의 주문 ID 열 유무 확인

In [4]:
def funnel_paths(parquet_dir, start, end):
    m = pd.read_parquet(parquet_dir / "mart_funnel_session.parquet", columns=["session_id", "e_created_at", "has_purchase", "sequence"])
    m["e_created_at"] = pd.to_datetime(m["e_created_at"], utc=True)
    p = m[m["e_created_at"].ge(start) & m["e_created_at"].lt(end) & m["has_purchase"].eq(1)]

    loop = "department → product → cart"
    home = "home → department → product → cart → purchase"
    repeated = {" → ".join([loop] * n + ["purchase"]) for n in (2, 3, 4)}
    path_type = p["sequence"].map(lambda s: "home_once" if s == home else ("repeat_loop_2_4" if s in repeated else "other"))
    t = path_type.value_counts().rename_axis("path_type").reset_index(name="sessions")
    t["purchase_session_share_pct"] = t["sessions"] / len(p) * 100
    return t


def session_identity(parquet_dir, start, end):
    e = pd.read_parquet(parquet_dir / "df_e.parquet", columns=["session_id", "u_id", "e_created_at"])
    e["e_created_at"] = pd.to_datetime(e["e_created_at"], utc=True)
    e = e[e["session_id"].notna() & e["e_created_at"].le(OBSERVATION_END)]

    g = e.groupby("session_id").agg(first_at=("e_created_at", "min"), events=("e_created_at", "size"),
                                    id_events=("u_id", "count"), ids=("u_id", "nunique"))
    g = g[g["first_at"].ge(start) & g["first_at"].lt(end)]
    shape = pd.Series("mixed", index=g.index)
    shape[g["id_events"].eq(0)] = "id_missing_all_events"
    shape[g["id_events"].eq(g["events"]) & g["ids"].eq(1)] = "same_id_all_events"
    return shape.value_counts().rename_axis("session_id_shape").reset_index(name="sessions")


def event_order_link(parquet_dir):
    # purchase 이벤트를 주문과 이을 열(주문 ID)이 이벤트 원본에 있는지 확인한다.
    cols = pq.read_schema(parquet_dir / "df_e.parquet").names
    linked = [c for c in cols if "order" in c.lower() or c.lower() == "o_id"]
    return pd.DataFrame([{"event_columns": " ".join(cols), "order_id_columns": " ".join(linked) or "none"}])

In [5]:
purchase_paths_table = funnel_paths(parquet_dir, start, end)
purchase_paths_table

,path_type,sessions,purchase_session_share_pct
0,repeat_loop_2_4,63993,51.877103
1,home_once,59362,48.122897


In [6]:
session_identity_table = session_identity(parquet_dir, start, end)
session_identity_table

,session_id_shape,sessions
0,id_missing_all_events,209466
1,same_id_all_events,123355


In [7]:
event_order_link_table = event_order_link(parquet_dir)
event_order_link_table

,event_columns,order_id_columns
0,e_id u_id session_id sequence_number e_created...,none


## 상품: 아우터 브랜드의 판매액과 판매 건수

* Outerwear & Coats의 Complete 품목을 분모로 브랜드 비중 비교
* The North Face 내부의 최빈 판매가 비중 확인

In [8]:
def outerwear_brands(parquet_dir, start, end, category="Outerwear & Coats", brands=("The North Face", "Carhartt")):
    m = pd.read_parquet(parquet_dir / "mart_product_mix_base.parquet",
                        columns=["oi_id", "order_date", "item_revenue", "category", "brand", "order_status"])
    m["order_date"] = pd.to_datetime(m["order_date"], utc=True)
    c = m[m["order_status"].eq(PURCHASE_STATUS) & m["order_date"].ge(start) & m["order_date"].lt(end)
          & m["order_date"].le(OBSERVATION_END) & m["category"].eq(category)]

    t = c.groupby("brand").agg(sales=("item_revenue", "sum"), items=("oi_id", "size"), mean_sale_price=("item_revenue", "mean"))
    t["sales_share_pct"] = t["sales"] / c["item_revenue"].sum() * 100
    t["item_share_pct"] = t["items"] / len(c) * 100
    t = t.loc[list(brands)].reset_index()

    # The North Face는 판매가 구성까지 본다: 가장 많이 반복된 판매가와 그 판매액 비중
    tnf = c[c["brand"].eq("The North Face")]
    top_price = tnf["item_revenue"].round(2).value_counts().idxmax()
    top = tnf[tnf["item_revenue"].round(2).eq(top_price)]
    price = pd.DataFrame([{"brand": "The North Face", "sale_price": top_price, "items": len(top),
                           "sales_share_pct": top["item_revenue"].sum() / tnf["item_revenue"].sum() * 100}])
    return t[["brand", "sales_share_pct", "items", "item_share_pct", "mean_sale_price"]], price

In [9]:
brands_table, price_table = outerwear_brands(parquet_dir, start, end)
brands_table

,brand,sales_share_pct,items,item_share_pct,mean_sale_price
0,The North Face,7.436559,36,2.339181,471.493611
1,Carhartt,7.052902,129,8.382066,124.791317


In [10]:
price_table

,brand,sale_price,items,sales_share_pct
0,The North Face,903.0,14,74.479624
